In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "sarvamai/sarvam-translate"

print(" Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#  Fix PAD token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(" Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

#  LoRA Config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)

print("Loading dataset...")
dataset = load_dataset("json", data_files="eng_tel_.jsonl", split="train")

#  Formatting function for your dataset
def format_example(example):
    en = example["eng_Latn"].strip()
    te = example["tel_Telu"].strip()

    return {
        "text": f"""### Instruction:
Translate English to Telugu.

### Input:
{en}

### Output:
{te}{tokenizer.eos_token}
"""
    }

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print("Starting training...")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,

    #  OLD TRL requires this
    processing_class=tokenizer,

    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-5,
        max_steps=2000,
        fp16=True,
        logging_steps=25,
        output_dir="./sarvam_lora_eng_telugu",
        save_steps=500,
        report_to="none",

        #  IMPORTANT FIX
        dataset_text_field="text"
    )
)

trainer.train()

trainer.model.save_pretrained("sarvam_en_te_lora")
tokenizer.save_pretrained("sarvam_en_te_lora")

print("Training complete!")
